# TODO 6.7.2：Prompt 与确定性计算消融

在同一个 FP16 模型和 883 条 FinQA dev 上比较四种 Prompt 方案。few-shot 示例只从 calibration/train 读取；不要把 dev/test 答案放入 Prompt。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/quality_repair.json'
RESULT_DIR = PROJECT_DIR / 'out/quality_repair_6_7_2'
sys.path.insert(0, str(PROJECT_DIR))
from src.baseline.service import BaselineService
from src.data.finqa_assets import load_assets
from src.evaluation.quality_repair import run_quality_repair_experiment

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
dataset_manifest = json.loads((DATASET_DIR / 'manifest.json').read_text(encoding='utf-8'))
service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'quality_service.jsonl')
report = run_quality_repair_experiment(
    service,
    assets.quality_dev,
    dataset_manifest,
    quality_config,
    RESULT_DIR,
    calibration=assets.calibration,
)
print(json.dumps(report, ensure_ascii=False, indent=2))
del service
gc.collect()
torch.cuda.empty_cache()

Mounted at /content/drive


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{
  "schema_version": 1,
  "experiment_name": "finqa-fp16-quality-repair-v1",
  "created_at": "2026-10-04T17:23:10.995428+00:00",
  "dataset": {
    "role": "evaluation_dev",
    "size": 883,
    "version": "main",
    "sha256": "7ba8d507634daa266268414448444421a18868e926632d805949058118b8a6d8"
  },
  "variants": [
    {
      "variant": "zero_shot_final_answer",
      "output_mode": "numeric_text",
      "calculator_enabled": false,
      "prompt_suffix": "\nReturn only the final numeric answer with its unit. Do not explain the calculation.",
      "metrics": {
        "total": 883,
        "parsed": 24,
        "correct": 2,
        "parse_rate": 0.027180067950169876,
        "numeric_accuracy": 0.0022650056625141564
      },
      "structured_output_rate": 0.0,
      "calculator_use_rate": 0.0,
      "format_retry_count": 0,
      "duration_seconds": 81.27123097499998,
      "predictions_file": "zero_shot_final_answer/predictions.jsonl",
      "predictions_sha256": "65761a1858b4f2db